# E1 Misplacement Classifier: End-to-End Training & Artifact Pipeline

This notebook provides a self-contained, reproducible training workflow for the **E1 Misplacement Classifier**.

### Operational Purpose
E1 predicts the in-flight risk probability $P(\text{misplace}) \in [0, 1]$ that a shipment is delayed or disrupted during transit.

### Mandatory Target Definition
The model target is an **E1 prototype disruption/misplacement-risk proxy label** derived from multimodal logistics incident telemetry (`disruption_type > 0`). It represents operational disruption risk, **NOT** a verified physical lost-package scan.

### Configurable Environment
This notebook runs cleanly in **Google Colab** and local environments without hardcoded local machine paths. Configure `DATA_ROOT` as needed.

In [ ]:
# Step 1: Dependencies and Environment
import sys
import os
import glob
import json
import pickle
import joblib
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    log_loss, confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve

# Configurable paths (Colab friendly)
IN_COLAB = 'google.colab' in sys.modules
DATA_ROOT = os.environ.get('DATA_ROOT', 'data/raw')
ARTIFACT_DIR = 'artifacts/misplacement_classifier/v1'
PROCESSED_DIR = 'data/processed'

print(f'Colab: {IN_COLAB} | Python: {sys.version.split()[0]} | sklearn: {sklearn.__version__}')
print(f'DATA_ROOT: {DATA_ROOT}')

In [ ]:
# Step 2: Automated Dataset Discovery & Audit
tabular_files = []
for ext in ('*.csv', '*.csv.gz', '*.parquet'):
    tabular_files.extend(glob.glob(os.path.join(DATA_ROOT, '**', ext), recursive=True))

print(f'Found {len(tabular_files)} tabular files:')
for f in tabular_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f' - {f} ({size_mb:.2f} MB)')

primary_csv = os.path.join(DATA_ROOT, 'zenodo', 'supplychain_datasetV0.csv')
if not os.path.exists(primary_csv):
    # Fallback to any detected Zenodo file
    zenodo_matches = [f for f in tabular_files if 'supplychain' in f.lower()]
    if zenodo_matches:
        primary_csv = zenodo_matches[0]
print(f'Selected Primary Dataset: {primary_csv}')

In [ ]:
# Step 3: Canonical Feature Mapping & Proxy Target Construction
# Canonical features: route, hub, carrier, congestion_index, num_handoffs, sorting_method, weather_flag, hour_of_day
# Target: misplaced (0 = normal, 1 = disruption event)

df_raw = pd.read_csv(primary_csv, low_memory=False)
df_raw['timestamp_dt'] = pd.to_datetime(df_raw['timestamp'])
df_raw = df_raw.sort_values('timestamp_dt').reset_index(drop=True)

# Congestion Index: Bounded [0, 1] normalized operational score
yard_norm = np.clip(df_raw['yard_utilization_pct'] / 100.0, 0.0, 1.0)
gate_norm = np.clip((df_raw['gate_turn_time_min'] - 6.0) / (98.77 - 6.0), 0.0, 1.0)
dwell_norm = np.clip((df_raw['dwell_time_at_node_min'] - 3.0) / (100.89 - 3.0), 0.0, 1.0)
congestion_index = np.clip(0.40 * yard_norm + 0.30 * gate_norm + 0.30 * dwell_norm, 0.0, 1.0).round(4)

# Weather flag: Upper quartile empirical threshold (4.88 mm)
weather_flag = (df_raw['precip_mm'] >= 4.88).astype(int)

# E1 prototype disruption/misplacement-risk proxy label
misplaced = (df_raw['disruption_type'] > 0).astype(int)

canonical_df = pd.DataFrame({
    'route': df_raw['route_id'].astype(str),
    'hub': df_raw['facility_id'].astype(str),
    'carrier': df_raw['carrier_mode'].astype(str),
    'congestion_index': congestion_index,
    'num_handoffs': df_raw['stop_count_last_24h'].astype(int),
    'sorting_method': 'unknown',
    'weather_flag': weather_flag,
    'hour_of_day': df_raw['timestamp_dt'].dt.hour.astype(int),
    'misplaced': misplaced
})

os.makedirs(PROCESSED_DIR, exist_ok=True)
canonical_df.to_csv(os.path.join(PROCESSED_DIR, 'e1_training.csv'), index=False)
print(f'Canonical records: {len(canonical_df)}, Positive rate: {misplaced.mean():.4f}')
canonical_df.head(5)

In [ ]:
# Step 4: Chronological Temporal Split (70% Train / 15% Validation / 15% Test)
n = len(canonical_df)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_df = canonical_df.iloc[:n_train].copy()
val_df = canonical_df.iloc[n_train:n_train + n_val].copy()
test_df = canonical_df.iloc[n_train + n_val:].copy()

ts = df_raw['timestamp_dt']
print(f'Train: {len(train_df)} ({ts.iloc[0]} to {ts.iloc[n_train-1]}) | Pos: {train_df["misplaced"].mean():.4f}')
print(f'Val:   {len(val_df)} ({ts.iloc[n_train]} to {ts.iloc[n_train+n_val-1]}) | Pos: {val_df["misplaced"].mean():.4f}')
print(f'Test:  {len(test_df)} ({ts.iloc[n_train+n_val]} to {ts.iloc[-1]}) | Pos: {test_df["misplaced"].mean():.4f}')

In [ ]:
# Step 5: Fit Feature Preprocessing Pipeline
cat_features = ['route', 'hub', 'carrier', 'sorting_method']
num_features = ['congestion_index', 'num_handoffs', 'weather_flag', 'hour_of_day']
feature_cols = cat_features + num_features

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num', StandardScaler(), num_features)
    ]
)

# Fit strictly on training split
X_train = preprocessor.fit_transform(train_df[feature_cols])
y_train = train_df['misplaced'].values

X_val = preprocessor.transform(val_df[feature_cols])
y_val = val_df['misplaced'].values

X_test = preprocessor.transform(test_df[feature_cols])
y_test = test_df['misplaced'].values

print(f'Transformed dimensions: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}')

In [ ]:
# Step 6: Candidate Modeling & Validation Model Selection
candidates = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
}

val_results = {}
for name, clf in candidates.items():
    print(f'Fitting candidate: {name}...')
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_val)[:, 1]
    pred = (prob >= 0.5).astype(int)
    val_results[name] = {
        'roc_auc': float(roc_auc_score(y_val, prob)),
        'pr_auc': float(average_precision_score(y_val, prob)),
        'accuracy': float(accuracy_score(y_val, pred)),
        'precision': float(precision_score(y_val, pred, zero_division=0)),
        'recall': float(recall_score(y_val, pred, zero_division=0)),
        'f1': float(f1_score(y_val, pred, zero_division=0)),
        'brier_score': float(brier_score_loss(y_val, prob)),
        'log_loss': float(log_loss(y_val, prob))
    }

val_table = pd.DataFrame(val_results).T
print('\nValidation Comparison:')
print(val_table[['roc_auc', 'pr_auc', 'f1', 'brier_score', 'log_loss', 'accuracy']])

best_candidate = val_table['roc_auc'].idxmax()
print(f'\nSelected Best Candidate Model: {best_candidate}')

In [ ]:
# Step 7: Train Final Production Model & Evaluate Once on Held-out Test Set
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

prod_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num', StandardScaler(), num_features)
    ]
)

X_train_val = prod_preprocessor.fit_transform(train_val_df[feature_cols])
y_train_val = train_val_df['misplaced'].values

X_test_final = prod_preprocessor.transform(test_df[feature_cols])

prod_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
prod_model.fit(X_train_val, y_train_val)

# Single test evaluation
test_prob = prod_model.predict_proba(X_test_final)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

test_metrics = {
    'roc_auc': float(roc_auc_score(y_test, test_prob)),
    'pr_auc': float(average_precision_score(y_test, test_prob)),
    'accuracy': float(accuracy_score(y_test, test_pred)),
    'precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'recall': float(recall_score(y_test, test_pred, zero_division=0)),
    'f1': float(f1_score(y_test, test_pred, zero_division=0)),
    'brier_score': float(brier_score_loss(y_test, test_prob)),
    'log_loss': float(log_loss(y_test, test_prob))
}

print('Held-out Test Evaluation Results:')
for k, v in test_metrics.items():
    print(f'  {k:15s}: {v:.4f}')

In [ ]:
# Step 8: Diagnostic Curves
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# Confusion Matrix
cm = confusion_matrix(y_test, test_pred)
axes[0, 0].imshow(cm, cmap=plt.cm.Blues)
axes[0, 0].set_title('Confusion Matrix (Held-out Test)')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('True')
axes[0, 0].set_xticks([0, 1], ['Normal (0)', 'Disrupted (1)'])
axes[0, 0].set_yticks([0, 1], ['Normal (0)', 'Disrupted (1)'])
for i in range(2):
    for j in range(2):
        axes[0, 0].text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[0, 1].plot(fpr, tpr, color='darkorange', label=f'ROC (AUC = {test_metrics["roc_auc"]:.3f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend()

# PR Curve
prec, rec, _ = precision_recall_curve(y_test, test_prob)
axes[1, 0].plot(rec, prec, color='green', label=f'PR (AUC = {test_metrics["pr_auc"]:.3f})')
axes[1, 0].axhline(y=y_test.mean(), color='grey', linestyle='--', label=f'Baseline ({y_test.mean():.3f})')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()

# Calibration Curve
prob_true, prob_pred = calibration_curve(y_test, test_prob, n_bins=10)
axes[1, 1].plot(prob_pred, prob_true, 's-', color='blue', label=f'Model (Brier = {test_metrics["brier_score"]:.4f})')
axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1, 1].set_title('Calibration Curve')
axes[1, 1].set_xlabel('Mean Predicted Probability')
axes[1, 1].set_ylabel('Fraction of Positives')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Artifact Export & Reload Verification
os.makedirs(ARTIFACT_DIR, exist_ok=True)
model_file = os.path.join(ARTIFACT_DIR, 'model.pkl')
pipeline_file = os.path.join(ARTIFACT_DIR, 'feature_pipeline.joblib')

with open(model_file, 'wb') as f:
    pickle.dump(prod_model, f)
joblib.dump(prod_preprocessor, pipeline_file)
print(f'Exported model to {model_file} and pipeline to {pipeline_file}')

# Serialization reload test
del prod_model, prod_preprocessor

with open(model_file, 'rb') as f:
    reloaded_model = pickle.load(f)
reloaded_pipe = joblib.load(pipeline_file)

# Canonical inference contract test
canonical_input = {
    'route': 'R001',
    'hub': 'F001',
    'carrier': 'air',
    'congestion_index': 0.72,
    'num_handoffs': 4,
    'sorting_method': 'unknown',
    'weather_flag': 1,
    'hour_of_day': 19
}

df_in = pd.DataFrame([canonical_input])[feature_cols]
pred_prob = float(reloaded_model.predict_proba(reloaded_pipe.transform(df_in))[0, 1])
print(f'P(misplace) = {pred_prob:.4f}')
assert 0.0 <= pred_prob <= 1.0, 'Output probability out of bounds!'
print('[OK] Complete training and serialization verification passed!')